# LLM Generating Text

This notebook now contains complete worked solutions for the Topic 6 exercises on **Generating text**.

Each section includes:
- a direct answer to the written questions
- an executable reference implementation where appropriate
- small sanity checks against PyTorch behavior
- short practical notes about how decoding choices affect generation quality

The emphasis is on numerically correct probability computation and clear reasoning about autoregressive decoding.


In [ ]:
# Environment bootstrap for Google Colab and local notebooks.
# Run this cell first if the runtime is missing the assignment packages.

import sys
import subprocess

REQUIRED_PACKAGES = [
    "einops>=0.8",
    "einx>=0.4",
    "jaxtyping>=0.3",
    "numpy>=2.4",
    "psutil>=7",
    "pytest>=9.0",
    "pytest-timeout",
    "pypdf>=6.10.0",
    "regex>=2026.3.32",
    "tiktoken>=0.12.0",
    "torch~=2.11.0",
    "tqdm>=4.67",
    "wandb>=0.25",
    "ty>=0.0.26",
    "ruff>=0.15.8",
]

if "google.colab" in sys.modules:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *REQUIRED_PACKAGES])
else:
    print("Local runtime detected; using the currently selected kernel environment.")


In [ ]:
# Shared imports and helper snippets for Topic 6 exercises.

import math
from pathlib import Path

import torch
import torch.nn.functional as F

torch.manual_seed(0)

PROJECT_ROOT = Path.cwd()
print("Project root:", PROJECT_ROOT)
print("Torch version:", torch.__version__)

def print_shape(name, tensor):
    print(f"{name}: shape={tuple(tensor.shape)}, dtype={tensor.dtype}")


def stable_softmax(x: torch.Tensor, dim: int = -1) -> torch.Tensor:
    shifted = x - x.max(dim=dim, keepdim=True).values
    exp_shifted = torch.exp(shifted)
    return exp_shifted / exp_shifted.sum(dim=dim, keepdim=True)


def sample_next_token(logits: torch.Tensor, temperature: float = 1.0, top_k: int | None = None) -> tuple[int, torch.Tensor]:
    if temperature <= 0:
        raise ValueError("temperature must be positive")

    scaled = logits / temperature
    if top_k is not None:
        top_values, top_indices = torch.topk(scaled, k=top_k)
        filtered = torch.full_like(scaled, float("-inf"))
        filtered[top_indices] = scaled[top_indices]
        scaled = filtered

    probs = stable_softmax(scaled, dim=-1)
    token = int(torch.multinomial(probs, num_samples=1))
    return token, probs


def greedy_next_token(logits: torch.Tensor) -> int:
    return int(torch.argmax(logits, dim=-1))


## softmax

**Assignment description**

- Implement Section 6.0.0.1 **Softmax**.
- Match `tests/test_nn_utils.py::test_softmax_matches_pytorch`.
- Make the implementation numerically stable.
- Explain how logits become a probability distribution.

**What this is testing**

This topic checks whether you understand the last step before sampling. The model outputs logits, not probabilities, and softmax turns relative scores into a normalized distribution over the vocabulary.

**Pseudocode / attack plan**

```text
input logits tensor x
shift x by subtracting max(x) along the softmax dimension
exp_x = exp(shifted_x)
denominator = sum(exp_x) along the softmax dimension
probs = exp_x / denominator
return probs
```

**Implementation notes**

- Subtracting the maximum logit avoids overflow without changing the result.
- The output along the softmax dimension must sum to 1.

In [ ]:
# softmax completed solution

x = torch.tensor([
    [0.4655, 0.8303, 0.9608, 0.9656, 0.6840],
    [0.2583, 0.2198, 0.9334, 0.2995, 0.1722],
], dtype=torch.float32)

shifted = x - x.max(dim=-1, keepdim=True).values
probs = torch.exp(shifted) / torch.exp(shifted).sum(dim=-1, keepdim=True)
reference = torch.softmax(x, dim=-1)
custom = stable_softmax(x, dim=-1)

print_shape("x", x)
print("row sums:", probs.sum(dim=-1))
print(probs)
print("matches torch.softmax:", torch.allclose(custom, reference, atol=1e-7))
assert torch.allclose(probs, reference, atol=1e-7)
assert torch.allclose(custom, reference, atol=1e-7)

large_logits = torch.tensor([[1000.0, 1001.0, 999.0]], dtype=torch.float32)
print("stable_softmax on very large logits:", stable_softmax(large_logits, dim=-1))
print("torch.softmax on very large logits:", torch.softmax(large_logits, dim=-1))

answer_shift = "Subtracting the maximum logit does not change the final probabilities because softmax is invariant to adding or subtracting the same constant from every logit in a row: the common factor cancels in the numerator and denominator."
answer_compare = "The custom implementation matches torch.softmax(x, dim=-1) because both compute exponentials of shifted logits and then normalize them so each row sums to 1."
answer_distribution = "Softmax turns logits into a probability distribution by exponentiating relative scores to make them positive and then dividing by their sum so the probabilities across the vocabulary add up to 1."

print(answer_shift)
print(answer_compare)
print(answer_distribution)


## decoding

**Assignment description**

- Study Section 6.0.0.2 **Decoding**.
- Understand how a trained language model generates text one token at a time.
- Compare greedy decoding and probabilistic sampling.

**What this is testing**

This topic checks whether you understand autoregressive generation as a repeated loop: feed context in, read logits for the last position, choose the next token, append it, and continue.

**Pseudocode / attack plan**

```text
start with a prompt token sequence
repeat until max_new_tokens is reached:
    run model on the current context
    take logits from the last time step only
    convert logits to probabilities if sampling
    choose next token (greedy or sample)
    append next token to the sequence
return the full sequence
```

**Writeup guidance**

- Keep the distinction clear: training predicts all positions in parallel, decoding usually consumes one new token at a time.
- Always mention that only the final position's logits are used to choose the next token.

In [ ]:
# decoding completed solution

vocab = ["A", "B", "C", "D"]
prompt = [0, 1]

# Pretend these are the logits for the next token at each decoding step.
step_logits = [
    torch.tensor([3.0, 1.0, 0.2, -1.0]),
    torch.tensor([0.5, 2.2, 1.0, 0.0]),
    torch.tensor([0.1, 0.3, 3.4, 1.2]),
]

greedy_generated = prompt.copy()
for logits in step_logits:
    greedy_generated.append(greedy_next_token(logits))

print("greedy token ids:", greedy_generated)
print("greedy decoded:", " ".join(vocab[i] for i in greedy_generated))

torch.manual_seed(0)
sampled_generated = prompt.copy()
for logits in step_logits:
    next_token, probs = sample_next_token(logits)
    sampled_generated.append(next_token)
    print("step logits:", logits)
    print("step probs:", probs)
    print("sampled token:", next_token)

print("sampled token ids:", sampled_generated)
print("sampled decoded:", " ".join(vocab[i] for i in sampled_generated))

answer_last_step = "Generation uses only the final time-step logits because those are the model's predictions for the next token after seeing the entire current context; earlier positions correspond to older prediction targets that have already been consumed."
answer_loop = "Autoregressive decoding is a repeated loop: feed the current token sequence into the model, read the logits for the last position, choose one next token with either argmax or sampling, append it to the sequence, and then run the model again on the longer context."
answer_compare = "Greedy decoding always takes the highest-probability token and is deterministic, while probabilistic sampling draws from the full distribution and can produce more diverse but less predictable continuations."

print(answer_last_step)
print(answer_loop)
print(answer_compare)


## decoder_tricks

**Assignment description**

- Study Section 6.0.0.3 **Decoder tricks**.
- Experiment with temperature, top-k filtering, and deterministic vs stochastic decoding.
- Explain how these choices affect diversity, repetition, and coherence.

**What this is testing**

This topic checks whether you can reason about generation behavior, not just model architecture. Decoder settings can make the same model look sharp, boring, chaotic, or repetitive.

**Pseudocode / attack plan**

```text
given next-token logits:
    optionally divide logits by temperature
    optionally keep only the top-k logits and mask the rest
    softmax to get probabilities
    either choose argmax or sample from the filtered distribution
```

**Writeup guidance**

- Lower temperature sharpens the distribution.
- Higher temperature flattens it.
- Top-k limits the candidate set before sampling.

In [ ]:
# decoder_tricks completed solution

torch.manual_seed(0)

logits = torch.tensor([5.0, 4.0, 2.0, 0.5, -1.0])
temperature = 0.7
top_k = 3

scaled = logits / temperature
top_values, top_indices = torch.topk(scaled, k=top_k)
filtered = torch.full_like(scaled, float("-inf"))
filtered[top_indices] = scaled[top_indices]
probs = F.softmax(filtered, dim=-1)
sample = int(torch.multinomial(probs, num_samples=1))

print("scaled logits:", scaled)
print("top indices:", top_indices)
print("probs:", probs)
print("sampled token:", sample)

for temp in [0.2, 1.0, 2.0]:
    temp_probs = stable_softmax(logits / temp, dim=-1)
    print(f"temperature={temp} probs={temp_probs.tolist()}")

for k in [None, 2, 3]:
    torch.manual_seed(0)
    token, k_probs = sample_next_token(logits, temperature=1.0, top_k=k)
    print(f"top_k={k} probs={k_probs.tolist()} sampled={token}")

answer_temperature = "Lower temperature such as 0.2 sharpens the distribution and concentrates mass on the best token, while higher temperature such as 2.0 flattens the distribution and increases randomness by making lower-ranked tokens more competitive."
answer_topk = "Top-k changes the support of the distribution by setting every token outside the k highest logits to probability zero before sampling, so sampling can only choose from that restricted candidate set."
answer_greedy = "Greedy decoding is preferable when you want maximum determinism and consistency, such as evaluation, regression testing, or tasks where the safest next token is more important than diversity."
answer_tradeoff = "In practice, temperature and top-k trade off diversity against coherence: aggressive sharpening reduces randomness and repetition risk from bad low-probability tokens, while looser settings increase variety but can make output noisier or less stable."

print(answer_temperature)
print(answer_topk)
print(answer_greedy)
print(answer_tradeoff)
